# Notebook 03 — Event-Conditioned Reliability Analysis
**Thesis:** Chapter 7 | **Input:** `data/2_events.csv` | **Output:** `data/3_results.csv`

---

## Purpose — answering the three research questions

| RQ | Question | Analysis |
|----|---------|---------|
| RQ1 | What does reconstruction reveal about campaign-wide reliability? | §7.1 — global statistics, rolling PDR, burst distribution |
| RQ2 | How does reliability vary across environmental and operational events? | §7.3–7.6 — event-conditioned PDR for all 18 events |
| RQ3 | Which conditions are most strongly associated with loss risk? | §7.7 — logistic regression, odds ratios |

## Filtered dataset
All event-conditioned analysis uses the **radio-layer filtered dataset**:
```python
df_link = df[~df['is_outage'] & ~df['is_sf_artifact']]
```
This ensures that observed PDR variation is attributable to radio-channel conditions,  
not infrastructure downtime or SF rotation artifacts.

## 0 · Imports & Load

In [34]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / '2_events.csv')
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values(['device_id', 'time']).reset_index(drop=True)

# filtered dataset — radio-layer losses only (excludes outages and SF artifacts)
# TODO ....Tests unfiltered data.... df_link = df.copy().reset_index(drop=True)
df_link = df[~df['is_outage'] & ~df['is_sf_artifact']].copy().reset_index(drop=True)


print(f'Full dataset : {len(df):,} rows')
print(f'Filtered (df_link): {len(df_link):,} rows')
print(f'Excluded     : {len(df)-len(df_link):,} rows (outages + SF artifacts)')
print(f'Devices      : {sorted(df_link["device_id"].unique())}')

Full dataset : 1,217,313 rows
Filtered (df_link): 1,156,407 rows
Excluded     : 60,906 rows (outages + SF artifacts)
Devices      : ['ED0', 'ED1', 'ED2', 'ED3', 'ED4', 'ED5']


## 1 · Global Reconstruction Results  *(§7.1 — RQ1)*

~~Campaign~~-wide summary: how much was lost, and what kind of losses were they?

Three PDR values tell the decomposition story:
- **PDR_system** — application layer perspective (everything included)
- **PDR_raw** — radio layer, all losses
- **PDR_link** — radio layer, genuine environmental losses only

In [35]:
# global PDR decomposition
rx         = len(df)
lost_system = int(df['total_loss'].sum())
lost_raw    = int(df['mac_to_radio_loss'].sum())
lost_link   = int(df_link['mac_to_radio_loss'].sum())

pdr_system = rx / (rx + lost_system) * 100
pdr_raw    = rx / (rx + lost_raw)    * 100
pdr_link   = len(df_link) / (len(df_link) + lost_link) * 100

print('=== Campaign-Wide PDR Decomposition ===')
print(f'PDR_system (incl. app-MAC drops)     : {pdr_system:.2f}%')
print(f'PDR_raw    (radio, all losses)        : {pdr_raw:.2f}%')
print(f'PDR_link   (radio, env. losses only)  : {pdr_link:.2f}%')
print(f'Gap PDR_system → PDR_link            : {pdr_link-pdr_system:.2f} pp')

# zone explanations
zone_labels = {
    'no_loss'    : 'packet arrived after a clean interval — nothing lost before it',
    'sf_artifact': 'packet arrived after an SF rotation transition artifact',
    'radio_loss' : 'packet arrived after 1-9 genuine radio losses',
    'ambiguous'  : 'packet arrived after 10-59 losses — cause unknown',
    'outage'     : 'packet arrived after 60+ losses — gateway was down',
}

print(f'\n=== Loss Zone Breakdown ===')
print(f"  {'Zone':<15} {'Count':>12} {'Pct':>11}  {'Meaning'}")
print('-' * 90)
zone_counts = df['loss_zone'].value_counts()
for zone, cnt in zone_counts.items():
    label = zone_labels.get(zone, '')
    print(f'  {zone:<15}: {cnt:>12,}  {cnt/len(df)*100:8.2f}%  {label}')

=== Campaign-Wide PDR Decomposition ===
PDR_system (incl. app-MAC drops)     : 59.26%
PDR_raw    (radio, all losses)        : 65.83%
PDR_link   (radio, env. losses only)  : 96.36%
Gap PDR_system → PDR_link            : 37.10 pp

=== Loss Zone Breakdown ===
  Zone                   Count         Pct  Meaning
------------------------------------------------------------------------------------------
  no_loss        :    1,135,043     93.24%  packet arrived after a clean interval — nothing lost before it
  sf_artifact    :       60,803      4.99%  packet arrived after an SF rotation transition artifact
  radio_loss     :       20,866      1.71%  packet arrived after 1-9 genuine radio losses
  ambiguous      :          498      0.04%  packet arrived after 10-59 losses — cause unknown
  outage         :          103      0.01%  packet arrived after 60+ losses — gateway was down


## 2 · Per-Device PDR_Link Summary  *(Table 7.1)*

PDR_link per device — the metric used for all event-conditioned analysis.  
The near-uniform PDR across vastly different link budgets is the first evidence  
for the collision-dominated loss hypothesis.

In [36]:
print(f"{'Device':<8} {'Dist(m)':^8} {'CW':^4} {'WW':^4} {'Received':^10} "
      f"{'Link Lost':^11} {'PDR_link':^10}")
print('-' * 65)

pdrs = {}
for dev, g in df_link.groupby('device_id'):
    rx   = len(g)
    lost = int(g['mac_to_radio_loss'].sum())
    pdr  = rx / (rx + lost) * 100
    pdrs[dev] = pdr
    dist = int(g['distance'].iloc[0])
    cw   = int(g['c_walls'].iloc[0])
    ww   = int(g['w_walls'].iloc[0])
    print(f"{dev:<8} {dist:^8} {cw:^4} {ww:^4} {rx:^10,} {lost:^11,} {pdr:^10.2f}%")

print('-' * 65)
pdr_range = max(pdrs.values()) - min(pdrs.values())
print(f"PDR range across devices: {pdr_range:.2f} percentage points")
print(f"\nKey observation: {pdr_range:.2f} percentage points spread across devices with very different distances")
print(f"""
- link budgets (10m direct LoS to 40m through 4 walls) — consistent with collision-dominated losses rather than signal quality degradation.
- ED4 is 37m away through 5 walls. ED0 is 10m away with nothing in between. Signal quality theory predicts ED4 should have dramatically lower PDR.
- Instead the difference is 0.43 percentage points (96.39 - 95.96). That is essentially zero.
- If signal quality drove losses, you would see 10-20 percentage points difference.
- 0.80 pp across the whole deployment means something else entirely drives losses — and that something is shared channel collisions,
    which affect all devices equally regardless of their individual link budget.
"""
       )

Device   Dist(m)   CW   WW   Received   Link Lost   PDR_link 
-----------------------------------------------------------------
ED0         10     0    0    192,030      7,185      96.39   %
ED1         8      1    0    191,072      7,547      96.20   %
ED2         23     0    2    193,876      6,479      96.77   %
ED3         18     1    2    190,340      7,652      96.14   %
ED4         37     0    5    189,995      7,996      95.96   %
ED5         40     2    2    199,094      6,848      96.67   %
-----------------------------------------------------------------
PDR range across devices: 0.80 percentage points

Key observation: 0.80 percentage points spread across devices with very different distances

- link budgets (10m direct LoS to 40m through 4 walls) — consistent with collision-dominated losses rather than signal quality degradation.
- ED4 is 37m away through 5 walls. ED0 is 10m away with nothing in between. Signal quality theory predicts ED4 should have dramatically lower PDR

## 3 · Statistical Analysis Core Function  *(§5.5.1)*

All event-conditioned comparisons use the same statistical framework:

1. **PDR computation** — for each event condition, compute mean PDR
2. **Mann-Whitney U test** — nonparametric test on `total_tx` distributions  
   (chosen because loss values are discrete and non-normal)
3. **Effect size** — rank-biserial correlation r = 1 − 2U/(n₁n₂)  
   ranges from -1 (event group has lower PDR) to +1 (event group has higher PDR)
4. **Significance** — α = 0.05 two-sided

This function is called for every binary event comparison in sections 4–8.

In [37]:
def pdr_compare(df_in, event_col, group1_label, group0_label=None):
    """
    Compare PDR between two event groups.
    Returns dict with PDR values, Mann-Whitney U test, and effect size.
    """
    g1 = df_in[df_in[event_col] == group1_label]
    g0 = df_in[df_in[event_col] != group1_label] if group0_label is None \
         else df_in[df_in[event_col] == group0_label]

    def pdr(g):
        rx = len(g)
        lost = int(g['mac_to_radio_loss'].sum())
        return rx / (rx + lost) * 100 if (rx + lost) > 0 else 0

    pdr1, pdr0 = pdr(g1), pdr(g0)
    diff = pdr1 - pdr0

    # Mann-Whitney U on total_tx — tests whether loss distributions differ
    stat, p = stats.mannwhitneyu(
        g1['total_tx'].values, g0['total_tx'].values, alternative='two-sided'
    )
    n1, n0 = len(g1), len(g0)
    r = 1 - 2 * stat / (n1 * n0)   # rank-biserial correlation

    return {
        'group1': group1_label, 'n1': n1, 'pdr1': pdr1,
        'group0': group0_label or f'not {group1_label}', 'n0': n0, 'pdr0': pdr0,
        'diff_pp': diff, 'p_value': p, 'effect_r': r,
        'significant': p < 0.05
    }

def print_comparison(result, title=''):
    sig = '✓ SIGNIFICANT' if result['significant'] else '✗ not significant'
    print(f"  {title}")
    print(f"    {result['group1']:>20}: {result['pdr1']:.2f}%  (n={result['n1']:,})")
    print(f"    {result['group0']:>20}: {result['pdr0']:.2f}%  (n={result['n0']:,})")
    print(f"    Difference        : {result['diff_pp']:+.2f} pp")
    print(f"    p-value           : {result['p_value']:.4f}  {sig}")
    print(f"    Effect size (r)   : {result['effect_r']:.4f}")
    print()

print('Statistical framework ready.')
print('Mann-Whitney U + rank-biserial correlation for all event comparisons.')

Statistical framework ready.
Mann-Whitney U + rank-biserial correlation for all event comparisons.


## 4 · Temporal Event-Conditioned Reliability  *(§7.3 — RQ2)*

**Hypothesis:** Office hours and weekdays should show lower PDR than off-peak periods  
due to higher human activity and increased RF interference from electronic devices.

Temporal events: E2 (weekday), E4 (time-of-day), E5 (office hours), E6 (season)

// Extra notes
p-value answers one question: "Could this difference have happened by random chance?"
For E2 (weekday vs weekend):

You observed weekday PDR = 96.09%, weekend PDR = 97.03%
p = 0.0000 means: the probability that this 0.94 pp difference happened purely by random chance is less than 0.001%

The rule:

p < 0.05 → the difference is real, not random → ✓ SIGNIFICANT
p ≥ 0.05 → cannot rule out random chance → ✗ not significant

Why p = 0.0000 for almost everything?
Because your dataset has 1.2 million rows. With that many data points, even a difference of 0.01 pp would be statistically significant. A huge sample size makes the test extremely sensitive — it can detect even meaningless tiny differences.
This is exactly why you also need the effect size r — because p alone does not tell you if the difference actually matters in practice.
The two questions together:
p-valueEffect size rQuestionIs it real?Does it matter?E2 weekdayYes (p=0.0000)Barely (r=0.0024)E3 CO₂ risingYes (p=0.0000)Yes (r=0.3227)
For E2 — real but practically meaningless.
For E3 — real AND practically meaningful.
That is why you always report both.

In [51]:
print('=' * 60)
print('TEMPORAL EVENTS (§7.3)')
print('=' * 60)

#todo FIRST THING TO DO AT 1.30 PM!!!!!!!!!!!!1
#todo Switch position of E2 and E1 so that we do occupancy first as E1 and E2, then the weekday shit starts in temporal Event as E3

temporal_results = {}

# E2: weekday vs weekend
print('\nE2: Weekday vs Weekend')
r = pdr_compare(df_link, 'e2_is_weekday', 1, 0)
print_comparison(r, 'Weekday (1) vs Weekend (0)')
temporal_results['E2_weekday'] = r

# E4: time-of-day bands
print('E4: Time-of-Day PDR')
for band in ['night', 'morning', 'peak', 'evening']:
    g = df_link[df_link['e4_time_of_day'] == band]
    rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
    pdr = rx / (rx + lost) * 100
    print(f'    {band:<10}: {pdr:.2f}%  (n={rx:,})')

# E5: office hours binary
print('\nE5: Office Hours vs Off-Peak')
r = pdr_compare(df_link, 'e5_office_hours', 1, 0)
print_comparison(r, 'Office hours (1) vs off-peak (0)')
temporal_results['E5_office'] = r

# E6: season
print('E6: Season PDR')
for season in ['autumn', 'winter', 'spring']:
    g = df_link[df_link['e6_season'] == season]
    rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
    pdr = rx / (rx + lost) * 100
    print(f'    {season:<10}: {pdr:.2f}%  (n={rx:,})')

TEMPORAL EVENTS (§7.3)

E2: Weekday vs Weekend
  Weekday (1) vs Weekend (0)
                       1: 96.09%  (n=820,283)
                   not 1: 97.03%  (n=336,124)
    Difference        : -0.94 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0024

E4: Time-of-Day PDR
    night     : 96.83%  (n=338,998)
    morning   : 96.72%  (n=145,160)
    peak      : 95.68%  (n=381,259)
    evening   : 96.53%  (n=290,990)

E5: Office Hours vs Off-Peak
  Office hours (1) vs off-peak (0)
                       1: 95.36%  (n=338,049)
                   not 1: 96.78%  (n=818,358)
    Difference        : -1.42 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0048

E6: Season PDR
    autumn    : 96.70%  (n=328,214)
    winter    : 95.99%  (n=407,201)
    spring    : 96.45%  (n=420,992)


## 5 · Occupancy-Conditioned Reliability  *(§7.4 — RQ2)*

**Hypothesis:** Higher CO₂ (more people) → more RF interference → lower PDR.  
CO₂ is the primary occupancy proxy. This is the most important environmental event family.

In [39]:
print('=' * 60)
print('OCCUPANCY EVENTS — CO2 (§7.4)')
print('=' * 60)

occupancy_results = {}

# E1: CO2 tier — three-way comparison
print('\nE1: CO2 Tier PDR')
co2_pdrs = {}
for tier in ['background', 'moderate', 'high']:
    g = df_link[df_link['e1_co2_tier'] == tier]
    rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
    pdr = rx / (rx + lost) * 100
    co2_pdrs[tier] = pdr
    print(f'    {tier:<12}: {pdr:.2f}%  (n={rx:,})')

# key comparison: high vs background
print('\nE1: High vs Background CO2 (key comparison)')
r = pdr_compare(df_link, 'e1_co2_tier', 'high', 'background')
print_comparison(r, 'High CO2 vs Background')
occupancy_results['E1_high_vs_bg'] = r

# moderate vs background
r = pdr_compare(df_link, 'e1_co2_tier', 'moderate', 'background')
print_comparison(r, 'Moderate CO2 vs Background')
occupancy_results['E1_mod_vs_bg'] = r

# E3: CO2 rising
print('E3: CO2 Rising vs Stable')
r = pdr_compare(df_link, 'e3_co2_rising', 1, 0)
print_comparison(r, 'CO2 rising (1) vs stable (0)')
occupancy_results['E3_rising'] = r

print('''
E3 is very dominant because:
During the brief transition moments when occupancy is actively increasing, the radio channel
becomes extremely congested — probably because many devices activate simultaneously (laptops connecting,
phones reconnecting to WiFi, IoT sensors waking up) creating a sudden collision burst.
''')


OCCUPANCY EVENTS — CO2 (§7.4)

E1: CO2 Tier PDR
    background  : 96.84%  (n=597,487)
    moderate    : 96.30%  (n=416,678)
    high        : 94.57%  (n=142,242)

E1: High vs Background CO2 (key comparison)
  High CO2 vs Background
                    high: 94.57%  (n=142,242)
              background: 96.84%  (n=597,487)
    Difference        : -2.27 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0096

  Moderate CO2 vs Background
                moderate: 96.30%  (n=416,678)
              background: 96.84%  (n=597,487)
    Difference        : -0.54 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0024

E3: CO2 Rising vs Stable
  CO2 rising (1) vs stable (0)
                       1: 33.48%  (n=1,087)
                   not 1: 96.53%  (n=1,155,320)
    Difference        : -63.05 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.3227


E3 is very dominant because:
During the brief transition moments when

## 6a · Air Quality Events — PM2.5  *(§7.5.1 — RQ2)*

PM2.5 spikes mark activity bursts — cleaning, printing, cooking.  
Device-specific 90th percentile thresholds (E7) account for location differences:  
ED4 (server room) baseline = 0.79 µg/m³ vs ED5 (kitchen) = 6.28 µg/m³.

In [40]:
print('=' * 60)
print('AIR QUALITY EVENTS — PM2.5 (§7.5.1)')
print('=' * 60)

atm_results = {}

# E7: PM2.5 spike
print('\nE7: PM2.5 Spike (device-specific 90th percentile)')
r = pdr_compare(df_link, 'e7_pm25_spike', 1, 0)
print_comparison(r, 'PM2.5 spike (1) vs normal (0)')
atm_results['E7_spike'] = r

# E8: PM2.5 absolute tier
print('E8: PM2.5 Absolute Tier')
for tier in ['clean', 'moderate', 'elevated']:
    g = df_link[df_link['e8_pm25_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<10}: {pdr:.2f}%  (n={rx:,})')

AIR QUALITY EVENTS — PM2.5 (§7.5.1)

E7: PM2.5 Spike (device-specific 90th percentile)
  PM2.5 spike (1) vs normal (0)
                       1: 94.28%  (n=115,186)
                   not 1: 96.59%  (n=1,041,221)
    Difference        : -2.31 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0153

E8: PM2.5 Absolute Tier
    clean     : 96.65%  (n=799,081)
    moderate  : 95.99%  (n=343,389)
    elevated  : 89.36%  (n=13,937)


## 6b · Atmospheric Events  *(§7.5.2–7.5.3 — RQ2)*

E9 (pressure tier), E10 (pressure drop), E11 (humidity), E12 (temperature).

In [41]:
print('=' * 60)
print('ATMOSPHERIC EVENTS (§7.5.2–7.5.3)')
print('=' * 60)

# E9: pressure tier
print('\nE9: Pressure Tier PDR')
for tier in ['low', 'medium_low', 'medium_high', 'high']:
    g = df_link[df_link['e9_pressure_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<15}: {pdr:.2f}%  (n={rx:,})')

# E10: pressure drop
print('\nE10: Pressure Drop Event')
r = pdr_compare(df_link, 'e10_pressure_drop', 1, 0)
print_comparison(r, 'Pressure drop (1) vs stable (0)')
atm_results['E10_drop'] = r

# E11: humidity tier
print('E11: Humidity Tier PDR')
for tier in ['dry', 'normal', 'humid', 'very_humid']:
    g = df_link[df_link['e11_humidity_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<12}: {pdr:.2f}%  (n={rx:,})')

# E12: temperature tier
print('\nE12: Temperature Tier PDR')
for tier in ['cold', 'cool', 'warm', 'hot']:
    g = df_link[df_link['e12_temp_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<8}: {pdr:.2f}%  (n={rx:,})')

ATMOSPHERIC EVENTS (§7.5.2–7.5.3)

E9: Pressure Tier PDR
    low            : 95.99%  (n=289,146)
    medium_low     : 96.09%  (n=289,341)
    medium_high    : 96.56%  (n=289,222)
    high           : 96.80%  (n=288,698)

E10: Pressure Drop Event
  Pressure drop (1) vs stable (0)
                       1: 6.68%  (n=35)
                   not 1: 96.40%  (n=1,156,372)
    Difference        : -89.72 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.7072

E11: Humidity Tier PDR
    dry         : 96.29%  (n=826,351)
    normal      : 96.57%  (n=327,788)
    humid       : 92.01%  (n=2,268)

E12: Temperature Tier PDR
    cold    : 96.52%  (n=290,346)
    cool    : 96.28%  (n=288,132)
    warm    : 96.45%  (n=289,417)
    hot     : 96.18%  (n=288,512)


## 7 · Spreading Factor and ToA-Conditioned Reliability  *(§7.6 — RQ2)*

**Hypothesis:** Higher SF → longer ToA → wider collision window → lower PDR.  
SF is the strongest non-environmental predictor. This section also presents  
the novel **SF × CO₂ joint analysis** — the interaction that is this thesis's headline finding.

**Joint finding hypothesis:** During high-occupancy periods, the PDR gap between  
low-SF and high-SF transmissions should widen — because longer ToA collides  
disproportionately with the elevated traffic from multiple active users.


**DATASET SELECTION **EXPLANATION** FROM AI:**

Should you use full dataset with artifacts for SF analysis?
No. Using the full dataset would be scientifically wrong here.
The SF artifacts (60,803 rows) are losses caused by the SF rotation protocol — a firmware/transition issue. They are not caused by ToA-collision dynamics. If you include them, SF10 would look even worse — but for the wrong reason. You would be mixing two completely different phenomena:

SF10 loses more because longer ToA → more collisions (what you are studying); SF10 loses more because of rotation artifacts (firmware issue, not physics) Your current analysis correctly isolates finding 1 by removing the artifacts. The 6.71 pp drop SF7→SF10 in df_link is the pure ToA-collision effect.

_Evidence 1 — Deterministic pattern:_
Normal radio losses are random — you see loss=1, 2, 3, 5, 7 scattered across SF10. But loss=8 with app_drop=2 appears 52,672 times, always at SF10, always the exact same values. Random collisions do not produce identical loss counts every single time. A deterministic mechanism does.
_Evidence 2 — Predecessor SF analysis:_
99.97% of SF10 artifact intervals are immediately preceded by SF7. If these were random radio losses, you would expect the preceding SF to be distributed roughly equally across SF7, SF8, SF9, SF10. Instead it is almost exclusively SF7. The loss is triggered by the SF7→SF10 transition specifically.
_Evidence 3 — ToA jump:_
SF7→SF10 is a 6× jump in Time-on-Air (71.9ms→452.6ms). No other SF transition is that large. The severity of the artifact scales with ToA jump size — SF9 has a smaller artifact, SF7 and SF8 have none.

**More Explanation from AI:**

What physically happens during SF7→SF10 transition:
When the firmware switches from SF7 to SF10, the radio chip must completely reconfigure:

Symbol duration changes from 1.024ms to 16.384ms (16× longer)
The receiver sensitivity threshold changes
The demodulator parameters change
The frequency hopping pattern may reset

During this reconfiguration window, the firmware is in a transitional state. Two things happen simultaneously:
The 2 internal drops (app_to_mac_drop=2): The application layer keeps generating packets during reconfiguration. The MAC layer is busy reconfiguring the radio chip and cannot accept them. They are dropped internally before the radio fires.
The 8 radio losses (mac_to_radio_loss=8): The radio starts firing with SF10 parameters but the gateway cannot yet decode them properly. This could be because:

The gateway needs a few packets to synchronise to the new SF10 symbol timing
The first few SF10 transmissions use incorrect preamble settings during the transition
The radio chip is still stabilising its oscillator for the longer symbol duration

The gateway hears something but cannot decode it — so f_count increments on the device but nothing arrives at the gateway.

Why exactly 8 and 2 every single time:
The reconfiguration takes a fixed amount of time — it is deterministic firmware behaviour. The device transmits every 60 seconds. The reconfiguration window is always the same length. So the number of packets lost during that window is always the same — 8 radio losses and 2 internal drops, every single transition, without exception.
This is what qualifies it as an artifact rather than random radio loss — it is deterministic, reproducible, and caused by the measurement protocol, not the radio environment.

In [42]:
print('=' * 60)
print('SF AND TOA EVENTS (§7.6)')
print('=' * 60)

sf_results = {}

# E13: PDR per individual SF value
print('\nE13: PDR by Individual Spreading Factor')
sf_pdrs = {}
for sf in [7, 8, 9, 10]:
    g = df_link[df_link['e13_sf'] == sf]
    rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
    pdr = rx / (rx + lost) * 100
    sf_pdrs[sf] = pdr
    toa = {7:71.9, 8:133.6, 9:246.8, 10:452.6}[sf]
    print(f'    SF{sf} (ToA={toa}ms): {pdr:.2f}%  (n={rx:,})')

print(f'\n    PDR drop SF7→SF10: {sf_pdrs[7]-sf_pdrs[10]:.2f} pp')

# E14: SF tier binary
print('\nE14: SF Tier Comparison')
r = pdr_compare(df_link, 'e14_sf_tier', 'high_sf', 'low_sf')
print_comparison(r, 'High SF (9-10) vs Low SF (7-8)')
sf_results['E14_tier'] = r

# E14 x E1: JOINT ANALYSIS — SF tier x CO2 tier
print('\n' + '=' * 60)
print('NOVEL JOINT ANALYSIS: SF Tier × CO2 Tier  (Table 7.7)')
print('=' * 60)
print(f"{'CO2 Tier':<15} {'Low SF (7-8)':>14} {'High SF (9-10)':>16} {'Gap (pp)':>10}")
print('-' * 58)

joint_results = {}
for co2_tier in ['background', 'moderate', 'high']:
    g_low  = df_link[(df_link['e14_sf_tier']=='low_sf')  & (df_link['e1_co2_tier']==co2_tier)]
    g_high = df_link[(df_link['e14_sf_tier']=='high_sf') & (df_link['e1_co2_tier']==co2_tier)]

    def pdr(g):
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        return rx / (rx + lost) * 100 if (rx+lost) > 0 else 0

    pdr_low  = pdr(g_low)
    pdr_high = pdr(g_high)
    gap      = pdr_low - pdr_high
    joint_results[co2_tier] = {'low_sf': pdr_low, 'high_sf': pdr_high, 'gap': gap}
    print(f"    {co2_tier:<15} {pdr_low:>10.2f}%   {pdr_high:>12.2f}%   {gap:>+8.2f} pp")


print('-' * 58)
gap_bg = joint_results['background']['gap']
gap_hi = joint_results['high']['gap']
print(f'\nInteraction: gap widens from {gap_bg:.2f} pp (background) to {gap_hi:.2f} pp (high CO2)')
print(f'Widening = {gap_hi-gap_bg:.2f} pp — SF×occupancy interaction confirmed.')

SF AND TOA EVENTS (§7.6)

E13: PDR by Individual Spreading Factor
    SF7 (ToA=71.9ms): 98.10%  (n=317,047)
    SF8 (ToA=133.6ms): 97.40%  (n=314,525)
    SF9 (ToA=246.8ms): 97.47%  (n=297,675)
    SF10 (ToA=452.6ms): 91.39%  (n=227,160)

    PDR drop SF7→SF10: 6.71 pp

E14: SF Tier Comparison
  High SF (9-10) vs Low SF (7-8)
                 high_sf: 94.74%  (n=524,835)
                  low_sf: 97.75%  (n=631,572)
    Difference        : -3.01 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0099


NOVEL JOINT ANALYSIS: SF Tier × CO2 Tier  (Table 7.7)
CO2 Tier          Low SF (7-8)   High SF (9-10)   Gap (pp)
----------------------------------------------------------
    background           98.09%          95.38%      +2.71 pp
    moderate             97.58%          94.78%      +2.79 pp
    high                 96.82%          91.97%      +4.86 pp
----------------------------------------------------------

Interaction: gap widens from 2.71 pp (background)

## 7b · E15 (ToA Class) + E16 (Burst Loss Distribution)  *(§7.6, §7.2)*

**E15:** ToA class is derived from SF — same binary split as E14, different framing.  
Included here for completeness and as a predictor in the logistic regression.

**E16:** Burst loss distribution — Table 7.2.  
E16 serves two roles: descriptive taxonomy (Table 7.2) and source of the  
burst-risk outcome variable in the logistic regression (§7.7).

In [43]:
# E15: Time-on-Air class — same binary as E14, expressed as collision risk
print('E15: ToA Class PDR')
r = pdr_compare(df_link, 'e15_toa_class', 'long_toa', 'short_toa')
print_comparison(r, 'Long ToA vs Short ToA')

# E16: Burst loss distribution — Table 7.2
# E16 is used here as a descriptive taxonomy and as the burst-risk outcome in §7.7
print('\n' + '=' * 60)
print('E16: BURST LOSS DISTRIBUTION  (Table 7.2)')
print('=' * 60)
order  = ['no_loss', 'isolated', 'small_burst', 'large_burst']
counts = df_link['e16_loss_type'].value_counts().reindex(order).fillna(0).astype(int)
pcts   = (counts / len(df_link) * 100).round(1)
print(f"{'Loss Type':<15} {'Count':>12} {'Pct':>8}")
print('-' * 38)
for ltype in order:
    print(f"{ltype:<15} {counts[ltype]:>12,} {pcts[ltype]:>7.1f}%")
print('-' * 38)
print(f"{'Total':<15} {len(df_link):>12,} {'100.0%':>8}")
burst_total = int(counts[['small_burst','large_burst']].sum())
print(f'\nIntervals with burst (≥3 losses): {burst_total:,}  ({pcts[["small_burst","large_burst"]].sum():.1f}%)')

E15: ToA Class PDR
  Long ToA vs Short ToA
                long_toa: 94.74%  (n=524,835)
               short_toa: 97.75%  (n=631,572)
    Difference        : -3.01 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : -0.0099


E16: BURST LOSS DISTRIBUTION  (Table 7.2)
Loss Type              Count      Pct
--------------------------------------
no_loss            1,135,043    98.2%
isolated              15,024     1.3%
small_burst            4,450     0.4%
large_burst            1,890     0.2%
--------------------------------------
Total              1,156,407   100.0%

Intervals with burst (≥3 losses): 6,340  (0.6%)


## 8 · Signal Context Events  *(§7.5 — RQ2)*

Do packets with weaker signal strength show higher loss rates?  
E17 (RSSI) and E18 (ESP) use the signal quality of the last received packet  
as a proxy for channel state during loss episodes.

In [49]:
print('=' * 60)
print('SIGNAL CONTEXT EVENTS (§7.5)')
print('=' * 60)

signal_results = {}

# E17: RSSI tier
print('\nE17: PDR by RSSI Tier')
for tier in ['weak', 'moderate', 'strong']:
    g = df_link[df_link['e17_rssi_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<10}: {pdr:.2f}%  (n={rx:,})')

r = pdr_compare(df_link, 'e17_rssi_tier', 'weak', 'strong')
print()
print_comparison(r, 'Weak vs Strong RSSI')
signal_results['E17_weak_vs_strong'] = r
print(''' No pattern at all. Weak and strong RSSI have almost identical PDR — difference of only 0.02 pp. The moderate tier actually has the highest PDR which makes no physical sense if signal strength drove losses.
Effect size r = 0.0048 — essentially zero. The p-value is significant only because of the huge sample size.
What this means: Signal strength does not predict packet loss in this deployment. A device receiving at -128 dBm (weak) loses packets at the same rate as a device receiving at -29 dBm (strong). This is powerful evidence for the collision hypothesis — if losses were caused by weak signals failing to decode, weak RSSI devices would have dramatically lower PDR. They do not.
''')

# E18: ESP tier
print('E18: PDR by ESP Tier')
for tier in ['low_esp', 'medium_esp', 'high_esp']:
    g = df_link[df_link['e18_esp_tier'] == tier]
    if len(g) > 0:
        rx = len(g); lost = int(g['mac_to_radio_loss'].sum())
        pdr = rx / (rx + lost) * 100
        print(f'    {tier:<12}: {pdr:.2f}%  (n={rx:,})')

r = pdr_compare(df_link, 'e18_esp_tier', 'low_esp', 'high_esp')
print()
print_comparison(r, 'Low ESP vs High ESP')
signal_results['E18_low_vs_high'] = r
print('''Even more striking — high ESP (best effective signal power) has the LOWEST PDR of the three tiers. This is the opposite of what signal-quality theory predicts.
Effect size r = 0.0039 — negligible.
What this means: ESP, which combines both RSSI and SNR into a single signal quality metric, also shows no relationship with loss. High quality signal does not protect against packet loss.
''')

SIGNAL CONTEXT EVENTS (§7.5)

E17: PDR by RSSI Tier
    weak      : 96.29%  (n=232,755)
    moderate  : 96.50%  (n=336,467)
    strong    : 96.31%  (n=587,185)

  Weak vs Strong RSSI
                    weak: 96.29%  (n=232,755)
                  strong: 96.31%  (n=587,185)
    Difference        : -0.02 pp
    p-value           : 0.0000  ✓ SIGNIFICANT
    Effect size (r)   : 0.0048

 No pattern at all. Weak and strong RSSI have almost identical PDR — difference of only 0.02 pp. The moderate tier actually has the highest PDR which makes no physical sense if signal strength drove losses.
Effect size r = 0.0048 — essentially zero. The p-value is significant only because of the huge sample size.
What this means: Signal strength does not predict packet loss in this deployment. A device receiving at -128 dBm (weak) loses packets at the same rate as a device receiving at -29 dBm (strong). This is powerful evidence for the collision hypothesis — if losses were caused by weak signals failing to

## 9 · Complete Event Summary Table  *(Tables 7.x)*

All 18 events ranked by absolute PDR difference.  
This table maps directly to your thesis Chapter 7 summary tables.

In [29]:
print('=' * 60)
print('ALL EVENTS — PDR SUMMARY (ranked by effect size)')
print('=' * 60)

all_results = []

# binary events — compare active vs inactive
binary_events = [
    ('E2',  'e2_is_weekday',   1, 0,            'Weekday vs Weekend'),
    ('E3',  'e3_co2_rising',   1, 0,            'CO2 rising vs stable'),
    ('E5',  'e5_office_hours', 1, 0,            'Office hours vs off-peak'),
    ('E7',  'e7_pm25_spike',   1, 0,            'PM2.5 spike vs normal'),
    ('E10', 'e10_pressure_drop',1, 0,           'Pressure drop vs stable'),
    ('E14', 'e14_sf_tier', 'high_sf','low_sf',  'High SF vs Low SF'),
    ('E15', 'e15_toa_class','long_toa','short_toa','Long ToA vs Short ToA'),
]

for eid, col, g1, g0, label in binary_events:
    r = pdr_compare(df_link, col, g1, g0)
    all_results.append({
        'Event': eid, 'Comparison': label,
        'PDR_active': r['pdr1'], 'PDR_inactive': r['pdr0'],
        'Diff_pp': r['diff_pp'], 'p_value': r['p_value'],
        'Effect_r': r['effect_r'], 'Significant': r['significant']
    })

# categorical events — compare extreme tiers
cat_events = [
    ('E1',  'e1_co2_tier',      'high',      'background', 'High CO2 vs Background'),
    ('E4',  'e4_time_of_day',   'peak',      'night',      'Peak vs Night'),
    ('E6',  'e6_season',        'winter',    'autumn',     'Winter vs Autumn'),
    ('E8',  'e8_pm25_tier',     'elevated',  'clean',      'Elevated vs Clean PM2.5'),
    ('E9',  'e9_pressure_tier', 'high',      'low',        'High vs Low Pressure'),
    ('E11', 'e11_humidity_tier','very_humid','dry',         'Very Humid vs Dry'),
    ('E12', 'e12_temp_tier',    'hot',       'cold',       'Hot vs Cold'),
    ('E13', 'e13_sf',           10,          7,            'SF10 vs SF7'),
    ('E17', 'e17_rssi_tier',    'weak',      'strong',     'Weak vs Strong RSSI'),
    ('E18', 'e18_esp_tier',     'low_esp',   'high_esp',   'Low vs High ESP'),
]

for eid, col, g1, g0, label in cat_events:
    r = pdr_compare(df_link, col, g1, g0)
    all_results.append({
        'Event': eid, 'Comparison': label,
        'PDR_active': r['pdr1'], 'PDR_inactive': r['pdr0'],
        'Diff_pp': r['diff_pp'], 'p_value': r['p_value'],
        'Effect_r': r['effect_r'], 'Significant': r['significant']
    })

results_df = pd.DataFrame(all_results).sort_values('Diff_pp')

print(f"{'Event':<6} {'Comparison':<35} {'PDR_active':>11} {'PDR_inactive':>13} {'Diff(pp)':>10} {'p-value':>9} {'r':>7} {'Sig':>5}")
print('-' * 100)
for _, row in results_df.iterrows():
    sig = '✓' if row['Significant'] else '✗'
    print(f"{row['Event']:<6} {row['Comparison']:<35} {row['PDR_active']:>10.2f}% "
          f"{row['PDR_inactive']:>12.2f}% {row['Diff_pp']:>+9.2f} "
          f"{row['p_value']:>9.4f} {row['Effect_r']:>7.4f} {sig:>5}")

ALL EVENTS — PDR SUMMARY (ranked by effect size)
Event  Comparison                           PDR_active  PDR_inactive   Diff(pp)   p-value       r   Sig
----------------------------------------------------------------------------------------------------
E11    Very Humid vs Dry                         0.00%        96.29%    -96.29       nan     nan     ✗
E10    Pressure drop vs stable                   6.68%        96.40%    -89.72    0.0000 -0.7072     ✓
E3     CO2 rising vs stable                     33.48%        96.53%    -63.05    0.0000 -0.3227     ✓
E8     Elevated vs Clean PM2.5                  89.36%        96.65%     -7.29    0.0000 -0.0268     ✓
E13    SF10 vs SF7                              91.39%        98.10%     -6.71    0.0000 -0.0195     ✓
E14    High SF vs Low SF                        94.74%        97.75%     -3.01    0.0000 -0.0099     ✓
E15    Long ToA vs Short ToA                    94.74%        97.75%     -3.01    0.0000 -0.0099     ✓
E7     PM2.5 spike vs nor

## 10 · Logistic Regression — Loss Risk Model  *(§7.7 — RQ3)*

**Research Question 3:** Which conditions are most strongly associated with loss risk?

The model predicts whether any loss occurred in an interval:
```
y_i = 1[mac_to_radio_loss > 0]
```

Predictors: all 18 events + device-ID fixed effects (ED0 as reference).  
**OR > 1** → increased loss probability when event is active.  
**95% CI from bootstrap (1000 samples, stratified by device)** — required for thesis Table 7.8.

In [30]:
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample

# binary outcome: did any loss occur?
df_link = df_link.copy()
df_link['loss_occurred'] = (df_link['mac_to_radio_loss'] > 0).astype(int)

# encode categorical event columns as dummies
feature_df = pd.get_dummies(df_link[[
    'e1_co2_tier', 'e2_is_weekday', 'e3_co2_rising',
    'e4_time_of_day', 'e5_office_hours', 'e6_season',
    'e7_pm25_spike', 'e8_pm25_tier',
    'e9_pressure_tier', 'e10_pressure_drop', 'e11_humidity_tier', 'e12_temp_tier',
    'e13_sf', 'e14_sf_tier', 'e15_toa_class',
    'e17_rssi_tier', 'e18_esp_tier',
    'device_id'
]], drop_first=True).astype(float)

X = feature_df.values
feat_names = feature_df.columns.tolist()
y = df_link['loss_occurred'].values

# fit with bootstrap 95% CI (1000 samples, stratified by device) — Table 7.8
def fit_lr_bootstrap(X, y, n_boot=1000):
    model = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0)
    model.fit(X, y)
    auc   = roc_auc_score(y, model.predict_proba(X)[:, 1])
    coefs = model.coef_[0]
    boot_coefs = []
    for _ in range(n_boot):
        X_b, y_b = resample(X, y, stratify=df_link['device_id'].values, random_state=None)
        m = LogisticRegression(max_iter=500, solver='lbfgs', C=1.0)
        m.fit(X_b, y_b)
        boot_coefs.append(m.coef_[0])
    boot_coefs = np.array(boot_coefs)
    ci_lo = np.exp(np.percentile(boot_coefs, 2.5,  axis=0))
    ci_hi = np.exp(np.percentile(boot_coefs, 97.5, axis=0))
    ors   = np.exp(coefs)
    result = pd.Series(ors, index=feat_names).sort_values(ascending=False)
    ci_df  = pd.DataFrame({'OR': ors, 'CI_lo': ci_lo, 'CI_hi': ci_hi}, index=feat_names)
    return result, ci_df, auc

print('Fitting loss risk model (1000 bootstrap samples) ...')
odds_ratios, ci_loss, auc_loss = fit_lr_bootstrap(X, y)
print(f'  AUC (loss risk) = {auc_loss:.3f}')

print(f'\n{"Predictor":<40} {"OR":>8} {"95% CI":>20} {"Direction":>14}')
print('-' * 88)
for feat, OR in odds_ratios.items():
    lo = ci_loss.loc[feat, 'CI_lo']
    hi = ci_loss.loc[feat, 'CI_hi']
    direction = '↑ higher risk' if OR > 1 else '↓ lower risk'
    print(f'{feat:<40} {OR:>8.4f} [{lo:.3f}, {hi:.3f}]{direction:>14}')

Fitting loss risk model (1000 bootstrap samples) ...
  AUC (loss risk) = 0.662

Predictor                                      OR               95% CI      Direction
----------------------------------------------------------------------------------------
e3_co2_rising                             37.5910 [25.384, 43.721] ↑ higher risk
e8_pm25_tier_elevated                      1.9935 [1.671, 2.423] ↑ higher risk
e10_pressure_drop                          1.8958 [1.488, 2.685] ↑ higher risk
e17_rssi_tier_weak                         1.6994 [1.477, 1.751] ↑ higher risk
e7_pm25_spike                              1.5517 [1.475, 1.696] ↑ higher risk
e13_sf                                     1.5448 [1.426, 1.636] ↑ higher risk
e11_humidity_tier_humid                    1.5097 [1.141, 1.985] ↑ higher risk
e1_co2_tier_high                           1.3585 [1.312, 1.488] ↑ higher risk
e6_season_spring                           1.2855 [1.200, 1.387] ↑ higher risk
e6_season_winter                

## 11 · Burst Loss Risk Model  *(§7.7 — RQ3)*

Same logistic regression framework but predicting **burst loss** (large_burst):  
```
y_i = 1[e16_loss_type == 'large_burst']
```

Burst-loss odds ratios are systematically larger than packet-loss odds ratios  
for the same events — confirming that high-occupancy and high-SF conditions  
not only increase loss probability but disproportionately increase burst severity.

In [1]:
print('=' * 60)
print('LOGISTIC REGRESSION — BURST RISK MODEL (§7.7)')
print('=' * 60)

# burst outcome: mac_to_radio_loss >= 3 (burst threshold B=3)
# E16 large_burst = loss >= 5, but thesis uses B=3 as burst definition
df_link['burst_occurred'] = (df_link['mac_to_radio_loss'] >= 3).astype(int)
y_burst = df_link['burst_occurred'].values

print('Fitting burst risk model (1000 bootstrap samples) ...')
or_burst, ci_burst, auc_burst = fit_lr_bootstrap(X, y_burst)
print(f'  AUC (burst risk) = {auc_burst:.3f}')

print(f'\n{"Predictor":<40} {"OR (loss)":>10} {"OR (burst)":>12} {"Burst/Loss ratio":>16}')
print('-' * 82)
for feat in odds_ratios.index:
    or_l = odds_ratios[feat]
    or_b = or_burst[feat]
    ratio = or_b / or_l if or_l != 0 else 0
    print(f'{feat:<40} {or_l:>10.4f} {or_b:>12.4f} {ratio:>16.2f}x')

LOGISTIC REGRESSION — BURST RISK MODEL (§7.7)


NameError: name 'df_link' is not defined

## 12 · Key Findings Summary  *(§7 summary)*

Consolidates all results into thesis-ready statements for Chapter 7.

In [18]:
print('=' * 60)
print('KEY FINDINGS SUMMARY')
print('=' * 60)

print('\n--- RQ1: Campaign-Wide Reliability ---')
print(f'PDR_link (true radio): {pdr_link:.2f}%')
print(f'PDR_raw  (all losses): {pdr_raw:.2f}%')
print(f'Decomposition gap    : {pdr_link-pdr_raw:.2f} pp')
print(f'PDR range across devices: {pdr_range:.2f} pp  → collision-dominated evidence')

print('\n--- RQ2: Event-Conditioned PDR ---')
print('Top findings from event analysis:')
top3 = results_df.head(3)
for _, row in top3.iterrows():
    sig = 'p<0.05' if row['Significant'] else 'n.s.'
    print(f"  {row['Event']}: {row['Comparison']} → {row['Diff_pp']:+.2f} pp  ({sig})")

print(f'\nSF×CO2 interaction:')
print(f'  Background CO2: gap = {joint_results["background"]["gap"]:.2f} pp')
print(f'  High CO2      : gap = {joint_results["high"]["gap"]:.2f} pp')
print(f'  Widening      : {joint_results["high"]["gap"]-joint_results["background"]["gap"]:.2f} pp → interaction confirmed')

print('\n--- RQ3: Strongest Predictors ---')
print('Top 5 loss-risk predictors (by OR):')
for feat, OR in odds_ratios.head(5).items():
    print(f'  {feat:<40}: OR = {OR:.4f}')
print('\nTop 5 burst-risk predictors (by OR):')
for feat, OR in or_burst.head(5).items():
    print(f'  {feat:<40}: OR = {OR:.4f}')

KEY FINDINGS SUMMARY

--- RQ1: Campaign-Wide Reliability ---
PDR_link (true radio): 96.36%
PDR_raw  (all losses): 65.83%
Decomposition gap    : 30.53 pp
PDR range across devices: 0.80 pp  → collision-dominated evidence

--- RQ2: Event-Conditioned PDR ---
Top findings from event analysis:
  E11: Very Humid vs Dry → -96.29 pp  (n.s.)
  E10: Pressure drop vs stable → -89.72 pp  (p<0.05)
  E3: CO2 rising vs stable → -63.05 pp  (p<0.05)

SF×CO2 interaction:
  Background CO2: gap = 2.71 pp
  High CO2      : gap = 4.86 pp
  Widening      : 2.15 pp → interaction confirmed

--- RQ3: Strongest Predictors ---
Top 5 loss-risk predictors (by OR):
  e3_co2_rising                           : OR = 37.5910
  e8_pm25_tier_elevated                   : OR = 1.9935
  e10_pressure_drop                       : OR = 1.8958
  e17_rssi_tier_weak                      : OR = 1.6994
  e7_pm25_spike                           : OR = 1.5517

Top 5 burst-risk predictors (by OR):
  e3_co2_rising                        

## 13 · Save Results

In [ ]:
# save complete results table
results_df.to_csv(DATA_DIR / '3_results.csv', index=False)

# save OR tables
or_table = pd.DataFrame({
    'predictor': odds_ratios.index,
    'OR_loss': odds_ratios.values,
    'OR_burst': [or_burst[f] for f in odds_ratios.index]
})
or_table.to_csv(DATA_DIR / '3_odds_ratios.csv', index=False)

# save joint SF x CO2 results
joint_df = pd.DataFrame(joint_results).T
joint_df.index.name = 'co2_tier'
joint_df.to_csv(DATA_DIR / '3_joint_sf_co2.csv')

print(f'Saved: {DATA_DIR}/3_results.csv')
print(f'Saved: {DATA_DIR}/3_odds_ratios.csv')
print(f'Saved: {DATA_DIR}/3_joint_sf_co2.csv')